# JAWAD HASSAN
# 2230-0035
# BS AI
# ML
# LAB 11
# CatBoost in Machine Learning

## Lab Task
## **Dataset (Anomaly detection)**
--> Use any dataset of your Choice (should be of Anomaly detection or Credit Card fraud) to apply CatBoost and compare its results with other Boosting techniques
Change learning rate
Try removing categorical feature

### Datasets Used:
[paysim fraud](https://www.kaggle.com/datasets/ealaxi/paysim1)

This dataset is extremely imbalanced.

The dataset contains over 6.3 million rows (transactions). Just like the Kaggle credit card dataset, it is highly imbalanced. Only about 0.13% of the transactions are actual fraud.

In [10]:
#libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from catboost import CatBoostClassifier
from sklearn.metrics import classification_report, f1_score


In [1]:
!pip install catboost

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 8.4 MB/s eta 0:00:00


In [3]:
from google.colab import files
files.upload()

Saving kaggle.json to kaggle.json


{'kaggle.json': b'{"username":"jawadhasuna","key":"c87e0fb7288844a3f52a7bd59e3af73e"}'}

In [4]:
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

In [5]:
!kaggle datasets download -d ealaxi/paysim1

Dataset URL: https://www.kaggle.com/datasets/ealaxi/paysim1
License(s): CC-BY-SA-4.0
100% 178M/178M [00:01<00:00, 156MB/s]



In [6]:
!unzip paysim1.zip

Archive:  paysim1.zip
  inflating: PS_20174392719_1491204439457_log.csv  


In [7]:
df = pd.read_csv('PS_20174392719_1491204439457_log.csv')
print(df.head())

   step      type    amount     nameOrig  oldbalanceOrg  newbalanceOrig  \
0     1   PAYMENT   9839.64  C1231006815       170136.0       160296.36   
1     1   PAYMENT   1864.28  C1666544295        21249.0        19384.72   
2     1  TRANSFER    181.00  C1305486145          181.0            0.00   
3     1  CASH_OUT    181.00   C840083671          181.0            0.00   
4     1   PAYMENT  11668.14  C2048537720        41554.0        29885.86   

      nameDest  oldbalanceDest  newbalanceDest  isFraud  isFlaggedFraud  
0  M1979787155             0.0             0.0        0               0  
1  M2044282225             0.0             0.0        0               0  
2   C553264065             0.0             0.0        1               0  
3    C38997010         21182.0             0.0        1               0  
4  M1230701703             0.0             0.0        0               0  


In [9]:
#preprocess for cat boost and xg boost

df['is_merchant_dest'] = df['nameDest'].str.startswith('M').astype(int)

df = df.drop(['nameOrig', 'nameDest', 'isFlaggedFraud'], axis=1)

X = df.drop('isFraud', axis=1)
y = df['isFraud']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

cat_features = ['type']

X_train_catboost = X_train.copy()
X_test_catboost = X_test.copy()

X_train_xgb_lgb = X_train.copy()
X_test_xgb_lgb = X_test.copy()

le = LabelEncoder()
X_train_xgb_lgb['type'] = le.fit_transform(X_train_xgb_lgb['type'])
X_test_xgb_lgb['type'] = le.transform(X_test_xgb_lgb['type'])

In [11]:

model = CatBoostClassifier(
    iterations=100,
    learning_rate=0.1,
    depth=6,
    scale_pos_weight=100,
    cat_features=cat_features,
    verbose=10
)

model.fit(X_train_catboost, y_train)

y_pred = model.predict(X_test_catboost)

print(classification_report(y_test, y_pred))

0:	learn: 0.6067772	total: 2.15s	remaining: 3m 32s
10:	learn: 0.2237977	total: 17.1s	remaining: 2m 18s
20:	learn: 0.1092947	total: 32.2s	remaining: 2m 1s
30:	learn: 0.0664058	total: 48.2s	remaining: 1m 47s
40:	learn: 0.0484836	total: 1m 3s	remaining: 1m 31s
50:	learn: 0.0385509	total: 1m 18s	remaining: 1m 15s
60:	learn: 0.0333335	total: 1m 33s	remaining: 59.6s
70:	learn: 0.0276885	total: 1m 47s	remaining: 44.1s
80:	learn: 0.0242602	total: 2m 2s	remaining: 28.8s
90:	learn: 0.0223928	total: 2m 17s	remaining: 13.6s
99:	learn: 0.0210084	total: 2m 30s	remaining: 0us
              precision    recall  f1-score   support

           0       1.00      1.00      1.00   1270881
           1       0.26      0.97      0.42      1643

    accuracy                           1.00   1272524
   macro avg       0.63      0.99      0.71   1272524
weighted avg       1.00      1.00      1.00   1272524



In [13]:
#change learn rate and scale_pos_weight and remove categorical features
X_train_no_cat = X_train_catboost.drop(['type'], axis=1)
X_test_no_cat = X_test_catboost.drop(['type'], axis=1)

model_no_cat = CatBoostClassifier(
    iterations=100,
    learning_rate=0.01,
    depth=6,
    scale_pos_weight=50,
    verbose=10,
    random_seed=42
)

model_no_cat.fit(X_train_no_cat, y_train)

y_pred_no_cat = model_no_cat.predict(X_test_no_cat)
print(classification_report(y_test, y_pred_no_cat))

0:	learn: 0.6860798	total: 904ms	remaining: 1m 29s
10:	learn: 0.5994004	total: 8.35s	remaining: 1m 7s
20:	learn: 0.5313103	total: 14.3s	remaining: 53.9s
30:	learn: 0.4726025	total: 21.8s	remaining: 48.5s
40:	learn: 0.4219111	total: 27.7s	remaining: 39.9s
50:	learn: 0.3782491	total: 35.2s	remaining: 33.8s
60:	learn: 0.3405094	total: 41.2s	remaining: 26.3s
70:	learn: 0.3074488	total: 48.7s	remaining: 19.9s
80:	learn: 0.2784122	total: 1m 1s	remaining: 14.3s
90:	learn: 0.2538288	total: 1m 7s	remaining: 6.67s
99:	learn: 0.2334567	total: 1m 14s	remaining: 0us
              precision    recall  f1-score   support

           0       1.00      1.00      1.00   1270881
           1       0.41      0.74      0.53      1643

    accuracy                           1.00   1272524
   macro avg       0.71      0.87      0.76   1272524
weighted avg       1.00      1.00      1.00   1272524



In [15]:
#xgboost
import xgboost as xgb
from sklearn.metrics import classification_report

xgb_model = xgb.XGBClassifier(
    n_estimators=100,
    learning_rate=0.01,
    max_depth=6,
    scale_pos_weight=100,
    eval_metric='logloss',
    random_state=42
)

xgb_model.fit(X_train_xgb_lgb, y_train)

y_pred_xgb = xgb_model.predict(X_test_xgb_lgb)

print(classification_report(y_test, y_pred_xgb))

              precision    recall  f1-score   support

           0       1.00      1.00      1.00   1270881
           1       0.47      0.85      0.61      1643

    accuracy                           1.00   1272524
   macro avg       0.74      0.92      0.80   1272524
weighted avg       1.00      1.00      1.00   1272524



### After comparing xgboost and catboost, cat boost recall is 97 percent where as xgboost recall is 85 percent, so catboost works better and is more accurate and faster